# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
if device == "cuda":
    !nvidia-smi

Fri Nov 27 15:50:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 What is a quantum computer? What can we do with quantum computers? How is a quantum computer different from a supercomputer?
And of course, how is it different from a classical computer? How can we build a quantum computer? What are the applications of quantum computing? What are the challenges in designing a quantum computer?
Quantum computing is a new form of computing that uses the principles of quantum mechanics to perform calculations.
Quantum computing is a new form of computing that uses the principles of quantum mechanics to perform calculations. Quantum computers are machines that use quantum systems to process information. They can solve problems that are too difficult for classical computers, and they can do it much faster.
The idea of quantum computing dates back to the 1980s.

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a rapidly evolving field that leverages the principles of quantum mechanics to perform calculations and sol

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [8]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can break it down into smaller, more manageable steps.

First, we can rewrite 24 as 20 + 4. This allows us to use the distributive property of multiplication over addition, which states that a × (b + c) = a × b + a × c.

So, we can rewrite the problem as:
15 × (20 + 4) = 15 × 20 + 15 × 4

Now, we can solve each part separately.

15 × 20 = 300 (since 15 × 2 = 30, and we have two 20s, so 30 × 10 = 300)

15 × 4 = 60 (since 15 × 2 = 30, and we have two 4s, so 30 × 2 = 60)

Now, we add the two results together:
300 + 60 = 360

Therefore, 15 × 24 = 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To solve this problem, we need to determine the amount of flour needed per cookie and then multiply that by the number of cookies we want to make.



## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [10]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [11]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [12]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [13]:
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [14]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [15]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 27 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## 7. Відповіді на теоретичні питання (пункти 11-21)

### 11. Як довго виконувалися клітинки ноутбуку `02_chat_templates`? Чи були проблеми з запуском, які?

На середовищі Google Colab з відеокартою Tesla T4 GPU (15.6 GB VRAM) встановлення пакетів та імпорт бібліотек зайняли 27 секунд, завантаження обох моделей (`SmolLM3-3B-Base` та `SmolLM3-3B`, кожна займає близько 6.15 GB у `float16`) - 3 хвилини, а виконання інференсу та робота з датасетами - близько 45-60 секунд (сумарно близько 4 хвилин).

Проблем із запуском не виникло завдяки використанню `dtype=torch.float16` (оскільки T4 не підтримує `bfloat16` апаратно) та встановленню `transformers>=4.55.0`. Повідомлення про `HF_TOKEN`, `clean_up_tokenization_spaces` та `rope_parameters` є стандартними інформаційними попередженнями бібліотеки.

### 12. Для чого змінювати параметр `add_generation_prompt` в методі `apply_chat_template()` при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?

Параметр `add_generation_prompt=False` використовується для форматування завершених діалогів (тренувальних та валідаційних даних SFT), де відповідь асистента вже присутня і закінчується закриваючим токеном (`<|im_end|>`). Натомість `add_generation_prompt=True` використовується для інференсу, коли останнє повідомлення належить користувачеві (`user`) і в кінець потрібно дописати відкриваючий заголовок відповіді асистента, щоб модель почала генерувати текст.

Визначити значення параметра за відформатованим текстом можна за кінцем рядка: якщо він закінчується відкриваючим тегом ролі асистента (`<|im_start|>assistant`, після якого у режимі `/no_think` також додається порожній блок `<think>\n\n</think>`) - було задано `add_generation_prompt=True`. Якщо рядок завершується закриваючим токеном останньої репліки (`<|im_end|>`) без нового тегу `assistant` - було задано `add_generation_prompt=False`.

### 13. Чим відрізняються ролі `system` від `assistant` в шаблонах розмов?

Роль `system` - це службові інструкції розробника (роль моделі, стиль відповідей, обмеження, режим міркування `/think` або `/no_think`), які визначають глобальні правила поведінки асистента в межах усього діалогу.

Роль `assistant` - це безпосередні репліки самої мовної моделі у відповідь на повідомлення користувача (`user`). Під час інференсу модель генерує токени саме в межах блоку `assistant`, а під час SFT-тренування функція втрат обчислюється на токенах цієї ролі.

### 14. Прокоментуйте вивід даних в секції 2 «Як шаблон розмови форматує діалог». Які секції було додано шаблонізатором розмов і для чого?

Шаблон розмови SmolLM3 перетворив список словників у формат ChatML (`<|im_start|>роль ... <|im_end|>`) і автоматично додав до блоку `system` секцію `## Metadata` з полями `Knowledge Cutoff Date: June 2025` (фіксує дату відсічення знань моделі), `Today Date: 27 November 2026` (передає поточну дату), `Reasoning Mode: /no_think` або `/think` (явно вказує режим міркувань), а також секцію `## Custom Instructions`.

Якщо системне повідомлення не передавалося (як у `simple_qa`), підставляється базова інструкція за замовчуванням (`You are a helpful AI assistant named SmolLM...`). У `REASONING_TASK` при виборі режиму `/think` шаблон автоматично додає розширену інструкцію щодо формату `<think> Thought section </think> Solution section`, а в режимі `/no_think` підставляє порожні теги `<think>\n\n</think>` після заголовка `assistant`.

### 15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.

Base-модель (`SmolLM3-3B-Base`) навчена виключно передбачати наступний токен у сирому тексті. Отримавши запит `"Explain me, what is quantum computing?"`, вона не відповідає на нього як співрозмовник, а генерує статистично правдоподібне продовження тексту (перелік інших схожих запитань про квантові комп'ютери та загальні енциклопедичні речення).

Instruct-модель (`SmolLM3-3B`), яка пройшла SFT на діалогах із розміткою ChatML, сприймає повідомлення користувача як інструкцію до виконання і одразу формує структуроване пояснення про принципи квантових обчислень, кубіти та суперпозицію.

### 16. Чому в режимі `/think` при генерації було передано параметр `max_new_tokens=300`?

У режимі `/think` модель спершу генерує внутрішній ланцюжок міркувань усередині блоку `<think> ... </think>`, де аналізує задачу, виконує проміжні обчислення і перевіряє себе, і лише після закриваючого тегу `</think>` виводить фінальну відповідь.

Оскільки розгорнуті міркування займають багато токенів, меншого ліміту (`max_new_tokens=150`) точно не вистачило б. Як видно з виводу секції 4, навіть при `max_new_tokens=300` модель витратила весь бюджет токенів на детальну самоперевірку всередині `<think>`, тому для завершеної фінальної відповіді в режимі `/think` потрібен ще більший запас токенів.

### 17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?

Модель правильно виконала обчислення для всіх трьох арифметичних задач ($15 \times 24 = 360$, 5 склянок борошна на 30 печив, решта $18.95). У режимі `/no_think` вона одразу вивела повні та завершені відповіді, а в режимі `/think` дійшла до правильних проміжних результатів усередині міркувань, але не встигла закрити блок `</think>` через обмеження `max_new_tokens=300`.

Для складних математичних задач, написання коду або багатокрокового логічного аналізу я обрав би режим `/think` (із достатнім лімітом `max_new_tokens`, наприклад 1024), а для простих арифметичних обчислень, швидких запитань чи перекладу - `/no_think`, щоб зекономити час та токени.

### 18. Які відмінності в шаблонах розмов Ви помітили в секції 5?

Модель `HuggingFaceTB/SmolLM3-3B` використовує теги `<|im_start|>role` і `<|im_end|>`, автоматично додає блок `## Metadata` у `system` та керує міркуваннями текстовими прапорцями `/think` або `/no_think` у системному повідомленні.

Модель `Qwen/Qwen3-0.6B` також використовує формат ChatML (`<|im_start|>role ... <|im_end|>`), але не вставляє метаданих із датами в `system`; режим міркувань керується булевим аргументом `enable_thinking` в `apply_chat_template` (при `enable_thinking=False` автоматично додає порожній блок `<think>\n\n</think>\n\n` після заголовка `assistant`).

Модель `microsoft/Phi-3.5-mini-instruct` використовує власні спеціальні токени ролей `<|system|>`, `<|user|>`, `<|assistant|>` із єдиним закриваючим токеном `<|end|>` після кожної репліки та `<|endoftext|>` у кінці, без метаданих і без механізму тегів `<think>`.

### 19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

Мовна модель сприймає структуру діалогу виключно через ті спеціальні токени, на яких її вчили. Якщо під час SFT або подальшого інференсу використати невідповідний шаблон чи сирий текст, модель не розпізнає межі реплік і ролей, через що якість генерації різко впаде.

Крім того, при донавчанні base-моделі потрібно явно призначити їй шаблон (наприклад, через `clone_chat_template`), а при донавчанні гібридних моделей (`Qwen3`, `SmolLM3`) - узгодити прапорці режиму міркувань (`enable_thinking`) із наявністю або відсутністю блоків `<think>` в еталонних відповідях датасету.

### 20. З яких даних було побудовано датасет `HuggingFaceTB/smoltalk2`?

`HuggingFaceTB/smoltalk2` - це великий відкритий інструкційний датасет від Hugging Face, створений для SFT і DPO моделей SmolLM3, у конфігурації SFT якого міститься 25 спеціалізованих піднаборів (splits).

Він об'єднує повсякденні розмови, математику, наукові задачі, виконання складних інструкцій, виклик зовнішніх інструментів (function/tool calling), роботу з таблицями та багатомовні дані. Значна частина відповідей у ньому є синтетичними даними, згенерованими сильнішими моделями (наприклад, `Qwen3-32B`) з подальшою фільтрацією якості - окремо для режимів із міркуваннями (`_think`) та без них (`_no_think`).

### 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат messages»?

Вихідний датасет `openai/gsm8k` має дві текстові колонки: `question` і `answer`. Функція `to_messages` перетворює кожен рядок у список із трьох словників діалогу (`messages`): додає системну інструкцію (`role: "